In [2]:
# %%
"""
Manifest Deep Diff Probe (Dynamic Generation Comparison).
Recursively audits nested dicts and lists across trading_config, chunk_specs,
data_lineage, environment, and checkpoint champion training epochs between any two generations.
"""

import argparse
import json
from pathlib import Path
from typing import Any, Dict, Tuple
import torch


def load_manifest(gen: int) -> Dict[str, Any]:
    paths = [
        Path(f"output/canonical_anchors/run_metadata_gen{gen}.json"),
        Path(f"output/model_checkpoints/walk_forward_gen{gen}/run_metadata.json"),
        Path(f"output/run_metadata_gen{gen}.json"),
    ]
    for p in paths:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Could not find manifest for Gen {gen}")


def deep_diff(
    v1: Any,
    v2: Any,
    prefix: str = "",
    label1: str = "GenA",
    label2: str = "GenB",
) -> Dict[str, Dict[str, Any]]:
    """Recursively diffs arbitrary combinations of dicts, lists, and primitives."""
    diffs = {}

    if isinstance(v1, dict) and isinstance(v2, dict):
        all_keys = sorted(set(v1.keys()).union(set(v2.keys())))
        for k in all_keys:
            path = f"{prefix}.{k}" if prefix else str(k)
            diffs.update(
                deep_diff(
                    v1.get(k), v2.get(k), prefix=path, label1=label1, label2=label2
                )
            )

    elif isinstance(v1, list) and isinstance(v2, list):
        if len(v1) != len(v2):
            diffs[f"{prefix}.__len__"] = {label1: len(v1), label2: len(v2)}
        max_len = max(len(v1), len(v2))
        for i in range(max_len):
            val1 = v1[i] if i < len(v1) else "<MISSING>"
            val2 = v2[i] if i < len(v2) else "<MISSING>"
            path = f"{prefix}[{i}]"
            diffs.update(
                deep_diff(val1, val2, prefix=path, label1=label1, label2=label2)
            )

    else:
        if v1 != v2:
            diffs[prefix] = {label1: v1, label2: v2}

    return diffs


def get_target_generations() -> Tuple[int, int]:
    """Retrieves generation IDs from CLI arguments or prompts the user interactively."""
    parser = argparse.ArgumentParser(
        description="Deep diff manifest probe between two generations."
    )
    parser.add_argument(
        "gen1", type=int, nargs="?", help="First generation ID (e.g., 23)"
    )
    parser.add_argument(
        "gen2", type=int, nargs="?", help="Second generation ID (e.g., 24)"
    )

    # parse_known_args prevents notebook/interactive kernel flags (e.g. -f) from throwing errors
    args, _ = parser.parse_known_args()

    gen1 = args.gen1
    gen2 = args.gen2

    if gen1 is None:
        while True:
            try:
                gen1 = int(input("Enter base generation ID (e.g., 23): ").strip())
                break
            except ValueError:
                print("❌ Invalid input. Please enter an integer.")

    if gen2 is None:
        while True:
            try:
                gen2 = int(input("Enter comparison generation ID (e.g., 24): ").strip())
                break
            except ValueError:
                print("❌ Invalid input. Please enter an integer.")

    return gen1, gen2


def main():
    gen1, gen2 = get_target_generations()
    label1 = f"Gen{gen1}"
    label2 = f"Gen{gen2}"

    print("=" * 70)
    print(f"🔍 DEEP DIFF: GENERATION {gen1} vs GENERATION {gen2} CONFIGURATIONS")
    print("=" * 70)

    try:
        m1 = load_manifest(gen1)
        m2 = load_manifest(gen2)
    except FileNotFoundError as e:
        print(f"❌ Error loading manifests: {e}")
        return

    sections = ["trading_config", "chunk_specs", "data_lineage", "environment"]

    for section in sections:
        c1 = m1.get(section, {})
        c2 = m2.get(section, {})
        diffs = deep_diff(c1, c2, prefix=section, label1=label1, label2=label2)

        # Ignore non-functional noise
        meaningful_diffs = {
            k: v
            for k, v in diffs.items()
            if not any(
                skip in k.lower()
                for skip in ["timestamp", "mtime", "manifest_path", "cache_file_mtime"]
            )
        }

        if meaningful_diffs:
            print(f"\n⚡ Differences detected in [{section}]:")
            for k, v in meaningful_diffs.items():
                print(f"  • {k}:")
                print(f"      Gen {gen1} -> {v[label1]}")
                print(f"      Gen {gen2} -> {v[label2]}")
        else:
            print(
                f"✅ [{section}] is 100% bitwise IDENTICAL between Gen {gen1} and Gen {gen2}."
            )

    # Audit Checkpoint Champion Epochs
    print("\n" + "=" * 70)
    print("🔎 SAVED CHECKPOINTS CHAMPION EPOCHS AUDIT")
    print("=" * 70)

    for gen in [gen1, gen2]:
        ckpt_dir = Path(f"output/model_checkpoints/walk_forward_gen{gen}")
        if ckpt_dir.exists():
            ckpts = sorted(list(ckpt_dir.glob("*.pt")))
            print(f"\nGen {gen} Checkpoints ({len(ckpts)} files in {ckpt_dir.name}):")
            for cp in ckpts:
                try:
                    payload = torch.load(cp, map_location="cpu", weights_only=False)
                    ep = payload.get("epoch", "N/A")
                    fit = payload.get("val_fitness", "N/A")
                    sh = payload.get("val_sharpe", "N/A")
                    fit_str = f"{fit:.4f}" if isinstance(fit, float) else str(fit)
                    sh_str = f"{sh:.3f}" if isinstance(sh, float) else str(sh)
                    print(
                        f"  • {cp.name:32s} | Best Epoch: {ep:>2} | Val Fitness: {fit_str:>8} | Val Sharpe: {sh_str:>6}"
                    )
                except Exception as ex:
                    print(f"  • {cp.name:32s} | Error reading: {ex}")
        else:
            print(f"⚠️ Checkpoint directory not found: {ckpt_dir}")


if __name__ == "__main__":
    main()

🔍 DEEP DIFF: GENERATION 24 vs GENERATION 25 CONFIGURATIONS
✅ [trading_config] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [chunk_specs] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [data_lineage] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.
✅ [environment] is 100% bitwise IDENTICAL between Gen 24 and Gen 25.

🔎 SAVED CHECKPOINTS CHAMPION EPOCHS AUDIT

Gen 24 Checkpoints (15 files in walk_forward_gen24):
  • model_chunk0_s101_champion.pt    | Best Epoch: 58 | Val Fitness:   0.0929 | Val Sharpe:  1.248
  • model_chunk0_s42_champion.pt     | Best Epoch: 88 | Val Fitness:   0.3933 | Val Sharpe:  1.588
  • model_chunk0_s777_champion.pt    | Best Epoch: 59 | Val Fitness:   0.0256 | Val Sharpe:  1.068
  • model_chunk1_s101_champion.pt    | Best Epoch:  6 | Val Fitness:   0.2370 | Val Sharpe:  0.491
  • model_chunk1_s42_champion.pt     | Best Epoch: 13 | Val Fitness:   0.0839 | Val Sharpe:  0.366
  • model_chunk1_s777_champion.pt    | Best Epoch:  9 | Val Fitness: 